# Day 2 — QAT Pareto sweep

Run this notebook once per available GPU. In each copy, set `RUN_NAME`, `WEIGHT_BITS`, and `ACTIVATION_BITS` to one of W8A8, W6A6, W4A6, or W4A4. Each command reloads the immutable baseline and logs complete console output with `tee`.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
username, repo_name = 'AdiGiriIIT', 'CS6886--Assignment-2'
!git clone https://{token}@github.com/{username}/{repo_name}.git assignment-2
%cd assignment-2


In [ ]:
from pathlib import Path

BASELINE_SOURCE = '/kaggle/input/datasets/adityagirishep23b048/baseline/baseline.pt'  # adjust only if your dataset mount differs
DATA_DIR = '/kaggle/input/datasets/adityagirishep23b048/cifar-data'
EXPECTED_SHA256 = '02ff38ac832c9fa3d72cad3db375b1103991b64eaf417c796ab352b49fbaae3d'

!python -m pip install -q PyYAML matplotlib
!mkdir -p results/checkpoints results/logs experiments/sweeps
!cp $BASELINE_SOURCE results/checkpoints/baseline.pt
!sha256sum results/checkpoints/baseline.pt
cifar_dir = Path(DATA_DIR) / 'cifar-10-batches-py'
assert all((cifar_dir / name).is_file() for name in ['data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5', 'test_batch', 'batches.meta'])
print('Using CIFAR-10 at', cifar_dir)


In [ ]:
# Required correctness gates before consuming a full GPU run. pipefail preserves failures through tee.
!set -o pipefail; python -m unittest discover -s tests -v 2>&1 | tee results/logs/day2-correctness.log
!set -o pipefail; python -m src.evaluate --checkpoint results/checkpoints/baseline.pt --data-dir "$DATA_DIR" --device cuda 2>&1 | tee results/logs/day2-baseline-reconstruction.log
!set -o pipefail; nvidia-smi 2>&1 | tee results/logs/day2-gpu.log


In [ ]:
# Change these three values in each parallel notebook.
RUN_NAME = 'w8a8-seed6886'
WEIGHT_BITS = 8
ACTIVATION_BITS = 8
EPOCHS = 12

assert (WEIGHT_BITS, ACTIVATION_BITS) in {(8, 8), (6, 6), (4, 6), (4, 4)}
print(f'Launching {RUN_NAME}: W{WEIGHT_BITS}A{ACTIVATION_BITS}')


In [ ]:
# pipefail makes a failed training command fail the cell even though tee is used.
!set -o pipefail; python -m src.qat --checkpoint results/checkpoints/baseline.pt --data-dir "$DATA_DIR" --device cuda --weight-bits $WEIGHT_BITS --activation-bits $ACTIVATION_BITS --epochs $EPOCHS --run-name $RUN_NAME 2>&1 | tee results/logs/$RUN_NAME.log


In [ ]:
# Inspect, package, and download the immutable record immediately after a successful run.
!cat experiments/sweeps/$RUN_NAME/metrics.json
!sha256sum results/checkpoints/qat-$RUN_NAME-best.pt results/checkpoints/qat-$RUN_NAME-latest.pt | tee experiments/sweeps/$RUN_NAME/checkpoint_sha256.txt
!tar -czf $RUN_NAME-artifacts.tgz experiments/sweeps/$RUN_NAME results/logs/$RUN_NAME.log results/checkpoints/qat-$RUN_NAME-best.pt
from IPython.display import FileLink
FileLink(f'{RUN_NAME}-artifacts.tgz')
